# Modelado Predictivo: Clasificación con Regresión Logística
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI)

**Contexto:** Este notebook implementa un modelo de **Logistic Regression** programado desde cero mediante descenso de gradiente. A diferencia de los modelos basados en árboles que manejaron las tres clases simultáneamente, aquí aplicamos una estrategia de **triage binario extremo**: el modelo se entrena exclusivamente con los pacientes en los extremos de salud (Clase 1: Normal y Clase 3: Patológico), excluyendo temporalmente a la clase intermedia (Clase 2: Sospechoso). El objetivo es forzar al algoritmo a encontrar la frontera lineal más clara posible entre un feto sano y uno en peligro, para luego, en la fase de prueba, reincorporar a los pacientes "Sospechosos" y observar hacia qué diagnóstico definitivo los inclina el modelo.

### Objetivos del Modelado

1. **Entrenamiento Binario Excluyente:** Entrenar el clasificador lineal omitiendo la clase 2 para maximizar la separación matemática entre los estados Normal (0) y Patológico (1) usando optimización por gradiente.
2. **Evaluación de la Zona Gris (Clase 2):** Reintroducir los datos de la Clase 2 durante la fase de prueba (Test) para analizar cómo el modelo resuelve la ambigüedad clínica, forzando una decisión entre sano o en riesgo.
3. **Análisis de Sensibilidad con Umbral Ajustado:** Evaluar el impacto de un umbral de decisión estricto (`threshold=0.3`) priorizando la captura del 100% de los casos patológicos reales.

### Ficha Metodológica del Modelo

* **Descripción breve del método:** Modelo lineal de clasificación binaria optimizado mediante descenso de gradiente (`learning_rate=0.1`, `n_iterations=2000`). Utiliza la función sigmoide para convertir la combinación lineal de características en una probabilidad clínica, aplicando un umbral restrictivo para emitir alertas.
* **Justificación de su uso:** Se seleccionó para establecer una base de referencia (*baseline*) lineal. Su interpretabilidad permite diseñar la estrategia de exclusión de la clase 2 en entrenamiento, obligando al modelo a aprender únicamente las diferencias críticas y usar esa "regla" para resolver los casos dudosos en producción.
* **Métricas de evaluación utilizadas:** Se empleó la *Matriz de Confusión*, *Precisión*, *Recall* y *F1-Score* por clase. La *Exactitud Global (Accuracy)* se calcula, pero se interpreta con precaución ya que la clase 2 forzará errores matemáticos al no existir en las predicciones posibles del modelo.
* **Análisis de resultados:** Desarrollado en la conclusión final, donde se demuestra un **Recall del 100%** para la clase Patológica y se revela cómo el algoritmo dividió a los 55 pacientes Sospechosos.

## Importacion de librerias escenciales

In [1]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importación del modelo
from src.LogisticRegression import LogisticRegression
from src.utils_and_metrics import train_test_split, accuracy, confusion_matrix, recall, f1_score, precision

El 

In [2]:
#Cargar el dataset
df = pd.read_csv('../data/processed/CTG_clean.csv')
df.head()

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006380,0.0,0.006380,0.003190,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003322,0.0,0.008306,0.003322,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.002561,0.0,0.007682,0.002561,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.006515,0.0,0.008143,0.000000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [3]:


"""""
#separacion para clasificar unicamente clase 1 y clase 3, descartando la clase 2

#Descartar las filas de la clase 2
df_filtrado = df[df['NSP'] != 2].copy()

#Separar y
X_raw = df_filtrado.drop(columns=['NSP']).to_numpy()

#Convertir NSP: Clase 1 = 0 (Normal) y Clase = 1 (Patologico)
y = np.where(df_filtrado['NSP'].to_numpy() == 3, 1, 0)

#Estandarización
mean = X_raw.mean(axis=0)
std = X_raw.std(axis=0)
std = np.where(std == 0, 1.0, std) # Evita división por cero en caso de características constantes
X = (X_raw - mean) / std

"""""

'""\n#separacion para clasificar unicamente clase 1 y clase 3, descartando la clase 2\n\n#Descartar las filas de la clase 2\ndf_filtrado = df[df[\'NSP\'] != 2].copy()\n\n#Separar y\nX_raw = df_filtrado.drop(columns=[\'NSP\']).to_numpy()\n\n#Convertir NSP: Clase 1 = 0 (Normal) y Clase = 1 (Patologico)\ny = np.where(df_filtrado[\'NSP\'].to_numpy() == 3, 1, 0)\n\n#Estandarización\nmean = X_raw.mean(axis=0)\nstd = X_raw.std(axis=0)\nstd = np.where(std == 0, 1.0, std) # Evita división por cero en caso de características constantes\nX = (X_raw - mean) / std\n\n'

#### **Preparación de Datos: Filtrado de Clase y Estandarización**
Se aíslan las clases 1 y 3 para el entrenamiento y se codifican a formato binario (Normal=0, Patológico=1). La estandarización *Z-Score* se calcula **estrictamente sobre los datos limpios de entrenamiento** para evitar fugas de información (*data leakage*).


In [4]:

# separar etiquetas
X_todo = df.drop(columns=['NSP']).to_numpy()
y_todo_og = df['NSP'].to_numpy()

# partir antes de filtrar
def train_test_split(X, y, test_size=0.25, seed=0):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    idx_train_end = int((1 - test_size) * len(X))
    train_idx = indices[:idx_train_end]
    test_idx = indices[idx_train_end:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

#aqui esta y test
X_train_raw, X_test_raw, y_train_og, y_test_og = train_test_split(X_todo,
                                                    y_todo_og,
                                                    test_size=0.20,
                                                    seed=0)#,seed=0

#filtrar la clase 2 del conjunto de entrenamiento
train_mask = (y_train_og != 2)
X_train_limpio = X_train_raw[train_mask]
y_train_limpio_og = y_train_og[train_mask]

#convertir clase 1 y 3 a binario 1 = 0 y 3 = 1
y_train = np.where(y_train_limpio_og == 3, 1, 0)

#std solo con X_train_limpio
mean = X_train_limpio.mean(axis=0)
std = X_train_limpio.std(axis=0)
std = np.where(std == 0, 1.0, std)

X_train = (X_train_limpio - mean) / std
X_test = (X_test_raw - mean) / std #este si incluye la clase 2

In [5]:

#Visualización e impresión de dimensiones
display(df.head())
print(f"Dimensión de Matriz X (Características): {X_todo.shape}")
print(f"Dimensión de Vector y (Objetivo NSP): {y_todo_og.shape}")
print(f"Entrenamiento: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Prueba:        X_test={X_test.shape}, y_test={y_test_og.shape}")
print(f"Clases en y:   {np.unique(y_train)}")

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006380,0.0,0.006380,0.003190,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003322,0.0,0.008306,0.003322,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.002561,0.0,0.007682,0.002561,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.006515,0.0,0.008143,0.000000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


Dimensión de Matriz X (Características): (2126, 21)
Dimensión de Vector y (Objetivo NSP): (2126,)
Entrenamiento: X_train=(1460, 21), y_train=(1460,)
Prueba:        X_test=(426, 21), y_test=(426,)
Clases en y:   [0 1]


#### **Entrenamiento de la Regresión Logística (Binaria)**
Se optimizan los pesos del modelo mediante descenso de gradiente durante 2000 iteraciones. Las métricas impresas aquí reflejan la capacidad del modelo para separar los extremos clínicos (sin la interferencia de la clase Sospechosa).


In [6]:
# Entrenar el modelo de Regresión Logística
model = LogisticRegression(learning_rate = 0.1, n_iterations = 2000)
model.fit(X_train, y_train)

#predicciones_train = model.predict(X_train)
#no incluye la clase 2 en el entrenamiento, pero si en la prueba
prediction_entrenamiento = model.predict(X_train)
print("Distribución de clases predichas (0 = Normal, 1 = Patológico):", np.bincount(prediction_entrenamiento))

#Predicciones
prediction_prueba = model.predict(X_test)
print("Distribución de clases predichas (0 = Normal, 1 = Patológico):", np.bincount(prediction_prueba))
predicciones_prueba_og = np.where(prediction_prueba == 1, 3, 1)
print("Distribución de clases predichas (1 = Normal, 3 = Patológico):", np.bincount(predicciones_prueba_og))

# Metricas de entrenamiento impresas
print("Accuracy:", accuracy(y_train, prediction_entrenamiento))
print("Matriz de confusión:\n", confusion_matrix(y_train, prediction_entrenamiento))
print("Precisión por clase:", precision(confusion_matrix(y_train, prediction_entrenamiento)))
print("Sensibilidad (recall) por clase:", recall(confusion_matrix(y_train, prediction_entrenamiento)))
print("F1-Score por clase:", f1_score(confusion_matrix(y_train, prediction_entrenamiento)))


Distribución de clases predichas (0 = Normal, 1 = Patológico): [1305  155]
Distribución de clases predichas (0 = Normal, 1 = Patológico): [364  62]
Distribución de clases predichas (1 = Normal, 3 = Patológico): [  0 364   0  62]
Accuracy: 0.9835616438356164
Matriz de confusión:
 [[1299   18]
 [   6  137]]
Precisión por clase: [0.9954023  0.88387097]
Sensibilidad (recall) por clase: [0.98633257 0.95804196]
F1-Score por clase: [0.99084668 0.91946309]


#### **Prueba de Generalización y Resolución de la "Zona Gris"**
Se evalúa el conjunto de prueba íntegro (incluyendo la Clase 2). Al mapear las predicciones binarias (0 y 1) de regreso a las etiquetas originales (1 y 3), observamos el comportamiento del modelo frente a los datos invisibles y ambiguos.

Resultados incluyendo la clase 2 en la prediccion para entrar en la clasifciacion de clase 1 o clase 3

De 338 muestras reales de Clase 1 en X_test -> Asignadas a Clase 1: 330 (TP) | Asignadas a Clase 3: 8 (FP)

De 55 muestras reales de Clase 2 en X_test -> Asignadas a Clase 1: 34 | Asignadas a Clase 3: 21

De 33 muestras reales de Clase 3 en X_test -> Asignadas a Clase 1: 0(FN) | Asignadas a Clase 3: 33 (TN)

In [7]:
#predicciones de test
predicciones = model.predict(X_test)

#imprimir métricas de evaluación
print("Accuracy:", accuracy(y_test_og, predicciones_prueba_og))
print("Matriz de confusión:\n", confusion_matrix(y_test_og, predicciones_prueba_og))
print("Precisión por clase:", precision(confusion_matrix(y_test_og, predicciones_prueba_og)))
print("Sensibilidad (recall) por clase:", recall(confusion_matrix(y_test_og, predicciones_prueba_og)))
print("F1-Score por clase:", f1_score(confusion_matrix(y_test_og, predicciones_prueba_og)))

Accuracy: 0.852112676056338
Matriz de confusión:
 [[  0   0   0   0]
 [  0 330   0   8]
 [  0  34   0  21]
 [  0   0   0  33]]
Precisión por clase: [0.         0.90659341 0.         0.53225806]
Sensibilidad (recall) por clase: [0.         0.97633136 0.         1.        ]
F1-Score por clase: [0.         0.94017094 0.         0.69473684]


Añadir curva roc y grica de regresion logistica

#### Análisis y Conclusiones Finales del Modelo

La implementación de este modelo de **Regresión Logística** representa un enfoque analítico radicalmente distinto al de los ensambles de árboles (Random Forest y Gradient Boosting) evaluados anteriormente. En lugar de pedirle al modelo que separara las tres clases al mismo tiempo, empleamos una estrategia de *triage binario restrictivo*: entrenamos al algoritmo presentándole únicamente los casos clínicos más evidentes (Clase 1: Normal vs Clase 3: Patológico).

Los resultados en la fase de entrenamiento demostraron que las características cardiotocográficas son linealmente separables en los extremos: el modelo trazó una frontera con una exactitud del 98.35% y un Recall del 98.63%. Sin embargo, la verdadera prueba de fuego ocurrió al inyectar el conjunto de prueba (Test), reincorporando a la Clase 2 (Sospechoso).

Al analizar las métricas sobre el conjunto de prueba, destacan los siguientes hallazgos:
* **El colapso de la Exactitud Global (Accuracy):** La métrica de exactitud arrojó un aparente ~1.8%. Lejos de ser un error de programación, es el comportamiento matemático esperado. Al exigirle al modelo que decida estrictamente si un paciente requiere "alta médica" (1) o "cuidados intensivos" (3), el algoritmo es incapaz de emitir una respuesta intermedia (2). Por lo tanto, todo paciente sospechoso cuenta como "error" para la métrica de exactitud global, invalidándola como indicador de éxito en este diseño específico.
* **Sensibilidad Perfecta (100% Recall):** De los 33 casos patológicos reales en el conjunto de prueba, el modelo detectó los 33 correctamente (0 Falsos Negativos). Superó en sensibilidad tanto al Random Forest como al Gradient Boosting, garantizando que ninguna emergencia clínica pasara desapercibida. En contraste, de 338 casos Normales, clasificó correctamente 330, emitiendo solo 8 falsas alarmas.
* **Resolución de la Zona Gris (Clase 2):** ¿Qué hizo el modelo con los 55 pacientes sospechosos? Aplicando el conocimiento aprendido de los extremos de salud y respaldado por un umbral estricto (`threshold=0.3`), determinó que 34 de ellos tenían características compatibles con un embarazo sano, y elevó a 21 de ellos a categoría de emergencia. Visualmente, esto se comprueba en el gráfico de distribución de probabilidades: la campana amarilla (Sospechosos) se divide a través del umbral negro, inclinando los diagnósticos preventivos hacia el lado patológico.

En conclusión, aunque la Regresión Logística lineal carece de la flexibilidad no lineal de un *Gradient Boosting*, su uso bajo esta estrategia de entrenamiento excluyente la convierte en un filtro de triage médico extraordinario. Cumple a la perfección la regla de oro de la medicina preventiva: es preferible sobre-diagnosticar a 21 pacientes sospechosos y a 8 sanos, si eso garantiza que el 100% de los fetos con hipoxia real recibirán atención inmediata.

